## **Cloning HF repo**

In [1]:
!git clone https://huggingface.co/datasets/JNIC1/HRVQA

Cloning into 'HRVQA'...
remote: Enumerating objects: 10134, done.
remote: Total 10134 (delta 0), reused 0 (delta 0), pack-reused 10134 (from 1)
Receiving objects: 100% (10134/10134), 1.93 MiB | 8.37 MiB/s, done.
Resolving deltas: 100% (19/19), done.
Updating files: 100% (10006/10006), done.
Filtering content: 100% (10003/10003), 15.74 GiB | 52.48 MiB/s, done.


## **Loading Training questions**

In [ ]:
import json
with open("/content/HRVQA/jsons/train_question.json", "r") as f:
  train_question = json.load(f)

In [ ]:
train_question['questions'][0]

{'image_id': 1,
 'question': 'What is the main transportation of this image?',
 'question_id': 0,
 'image_name': '2021_131000_460000_RGB_hrl_0_0',
 'question_type': 'transportation'}

In [ ]:
len(train_question['questions'])

540720

## **Loading Images**

In [ ]:
images=[]
import os
for img in os.listdir("/content/HRVQA/images"):
  images.append(int(img.split('.')[0]))


In [ ]:
len(images)

9999

In [ ]:
images[0:10]

[1274, 8768, 7377, 3501, 6057, 382, 7628, 4472, 3557, 6099]

## **Selecting only those questions whose image id is present in the list "images"**

In [ ]:
questions=[]
img_ids=[]
for item in train_question['questions']:
  if item['image_id'] in images:
    questions.append(item)
    img_ids.append(item['image_id'])
img_ids=list(set(img_ids))

In [ ]:
len(img_ids)

9999

In [ ]:
len(questions)

199980

In [ ]:
questions[199979]

{'image_id': 9999,
 'question': 'What sport can people do in this image?',
 'question_id': 199979,
 'image_name': '2021_132000_453000_RGB_hrl_11476_1024',
 'question_type': 'sports'}

## **Loading train_answer.json**

In [ ]:
with open("/content/HRVQA/jsons/train_answer.json", "r") as f:
  train_answer = json.load(f)

In [ ]:
train_answer['annotations'][0]

{'multiple_choice_answer': 'car', 'question_id': 0}

In [ ]:
len(train_answer['annotations'])

540720

## **Loading only those answers whose question_id is present in the list "questions"**

In [ ]:
question_ids_in_questions = {item['question_id'] for item in questions}
answers = []
for item in train_answer['annotations']:
  if item['question_id'] in question_ids_in_questions:
    answers.append(item)

In [ ]:
len(answers)

199980

In [ ]:
answers[199979]

{'multiple_choice_answer': 'aerobic', 'question_id': 199979}

### **Verifying Data Integrity**
Checking if all images have questions and all questions have answers.

In [ ]:
# Check if all image_ids from the file system have at least one question
question_image_ids = {q['image_id'] for q in questions}
missing_images = [img_id for img_id in images if img_id not in question_image_ids]

# Check if all question_ids in our filtered list have an answer
answer_question_ids = {a['question_id'] for a in answers}
missing_answers = [q['question_id'] for q in questions if q['question_id'] not in answer_question_ids]

print(f"Total unique images found: {len(images)}")
print(f"Images without questions: {len(missing_images)}")
print(f"Total questions filtered: {len(questions)}")
print(f"Questions without answers: {len(missing_answers)}")

if not missing_images and not missing_answers:
    print("\nSuccess: Data is consistent. Every image has questions and every question has an answer.")
else:
    if missing_images: print(f"First 5 missing images: {missing_images[:5]}")
    if missing_answers: print(f"First 5 missing answers (IDs): {missing_answers[:5]}")

Total unique images found: 9999
Images without questions: 0
Total questions filtered: 199980
Questions without answers: 0

Success: Data is consistent. Every image has questions and every question has an answer.


### **Downloading Processed Data**
Saving the filtered lists to JSON files and downloading them.

In [ ]:
import json
from google.colab import files

# Save the filtered questions
with open('filtered_questions.json', 'w') as f:
    json.dump(questions, f)

# Save the filtered answers
with open('filtered_answers.json', 'w') as f:
    json.dump(answers, f)

# Download the files
files.download('filtered_questions.json')
files.download('filtered_answers.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### **Zipping and Downloading Images**
Creating a zip archive of the dataset images for local download.

In [ ]:
!zip -r HRVQA_images.zip /content/HRVQA/images

Streaming output truncated to the last 5000 lines.
  adding: content/HRVQA/images/6657.png (deflated 2%)
  adding: content/HRVQA/images/1281.png (deflated 2%)
  adding: content/HRVQA/images/4401.png (deflated 2%)
  adding: content/HRVQA/images/4409.png (deflated 1%)
  adding: content/HRVQA/images/7688.png (deflated 2%)
  adding: content/HRVQA/images/8043.png (deflated 2%)
  adding: content/HRVQA/images/4353.png (deflated 2%)
  adding: content/HRVQA/images/7160.png (deflated 3%)
  adding: content/HRVQA/images/181.png (deflated 2%)
  adding: content/HRVQA/images/6696.png (deflated 2%)
  adding: content/HRVQA/images/4859.png (deflated 1%)
  adding: content/HRVQA/images/4267.png (deflated 2%)
  adding: content/HRVQA/images/7124.png (deflated 3%)
  adding: content/HRVQA/images/369.png (deflated 3%)
  adding: content/HRVQA/images/7575.png (deflated 2%)
  adding: content/HRVQA/images/7499.png (deflated 2%)
  adding: content/HRVQA/images/134.png (deflated 2%)
  adding: content/HRVQA/images/934

In [ ]:
from google.colab import files
files.download('HRVQA_images.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## **TODO: When training the model we need to train test split the Dataset**